# SPFI ×4 — test-set evaluation on EMPS (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/inference_colab.ipynb)

Loads weights from `train_colab.ipynb` and evaluates them on the **test split only** of EMPS (`test.csv`), using the repo's `algo/` code unchanged:

1. `algo.image_loading.load_image` + `algo.preprocessing.normalize_image` → HR image in [0, 1]
2. `algo.train.degrade` → LR image (the same blur + bicubic ×4 downscale as in training, sigma fixed per file)
3. the SPFI model (`algo.model.build_spfi`) → SR image
4. `algo.train.psnr(SR, HR)` and SSIM = `1 - algo.train.ssim_loss(SR, HR)` (the same SSIM as training validation)

The output is the **average PSNR and SSIM** over the test images, plus the model's **FLOPs**
(counted with `torch.utils.flop_counter`, like `SPFITrainer._gflops`).

**Why tiles:** the model uses global attention (cost grows with (H·W)²), so a whole image does not fit in
GPU memory. The LR image is cut into 64×64 tiles (the training patch size) that overlap by a few pixels,
each tile goes through the model, and the outputs are averaged back into one image.

**Before you start:** Runtime → Change runtime type → **GPU**.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


## Settings

In [ ]:
from pathlib import Path

# ---- Code (the algo/ package is used exactly as it is in the repo) ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "main"

# ---- Data: public EMPS release (same images and train/test split as NanoVision's datasets/emps/EMPS) ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"

WORK_DIR = Path("/content")
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"

# ---- Weights ----
# Path to the .pt file from train_colab.ipynb, e.g. on Drive:
#   "/content/drive/MyDrive/spfi_checkpoints/spfi_x4_best_<stamp>.pt"
# Leave empty to upload the file from your computer.
WEIGHTS_PATH = ""

# ---- Evaluation ----
TILE = 64            # LR tile size (= training LR patch size)
OVERLAP = 16         # LR pixels shared by neighbouring tiles
BATCH_TILES = 16     # tiles per forward pass; lower it if you run out of GPU memory
SHAVE = 4            # HR border pixels ignored by PSNR / SSIM (the usual "shave = scale"); 0 = whole image
SAVE_SR_IMAGES = False   # also save every SR image as a 16-bit TIFF in output/sr/


## Code and dependencies

In [ ]:
# torch / torchvision / numpy / Pillow are preinstalled on Colab; add what algo/ needs on top.
!pip install -q "pytorch_wavelets>=1.3" "PyWavelets>=1.6" "setuptools<81"


In [ ]:
import hashlib
import subprocess
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_split(split):
    """Extract train.csv, test.csv and only the images of `split` ('train' or 'test') into DATA_DIR/<split>/."""
    dest = DATA_DIR / split
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / f"{split}.csv").read_text())
        for image_id in ids:
            (dest / "images" / f"{image_id}.png").write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


import sys

if not CODE_DIR.exists():
    git("clone", "--depth", "1", "--branch", CODE_BRANCH, CODE_REPO_URL, str(CODE_DIR))
print("code:", git("log", "-1", "--format=%h %s", cwd=CODE_DIR).strip())

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))   # makes `algo` importable
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Dataset
EMPS is downloaded from its public release ([by256/emps](https://github.com/by256/emps), pinned commit),
so no GitHub token is needed. Its `train.csv` / `test.csv` are identical to the ones in NanoVision's
`datasets/emps/EMPS`.

Only the images listed in `test.csv` are extracted.

In [ ]:
EMPS_DIR, test_ids = extract_emps_split("test")
print(f"{len(test_ids)} test images in {EMPS_DIR / 'images'}")


## Load the weights

In [ ]:
if not WEIGHTS_PATH:
    from google.colab import files
    uploaded = files.upload()          # pick spfi_x4_best.pt
    WEIGHTS_PATH = str(WORK_DIR / next(iter(uploaded)))
elif WEIGHTS_PATH.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
from algo.model import build_spfi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Same checkpoint contract as the local app: {"model_state": ..., "hparams": {"scale", "embed_dim"}}
ckpt = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
hp = ckpt["hparams"]
SCALE = hp["scale"]
model = build_spfi(scale=SCALE, embed_dim=hp["embed_dim"])
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {Path(WEIGHTS_PATH).name}: hparams {hp}, iteration {ckpt.get('iteration', '?')}, "
      f"{n_params:,} parameters, device {device}")


## FLOPs

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


def count_flops(lr_h, lr_w):
    """FLOPs of one forward pass on a [1, 1, lr_h, lr_w] input."""
    with torch.no_grad(), FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 1, lr_h, lr_w, device=device))
    return counter.get_total_flops()


FLOPS_PER_TILE = count_flops(TILE, TILE)
print(f"{FLOPS_PER_TILE / 1e9:.3f} GFLOPs per {TILE}x{TILE} LR input "
      f"({TILE * SCALE}x{TILE * SCALE} output)")


## Evaluate on the test split

In [ ]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

from algo.image_loading import load_image
from algo.preprocessing import normalize_image
from algo.train import degrade, psnr, ssim_loss


def tile_starts(n):
    """Start offsets of TILE-sized windows covering 0..n, OVERLAP apart, the last one flush with the end."""
    if n <= TILE:
        return [0]
    starts = list(range(0, n - TILE, TILE - OVERLAP))
    return starts + [n - TILE]


@torch.no_grad()
def super_resolve(lr):
    """[1, 1, h, w] LR -> ([1, 1, h*SCALE, w*SCALE] SR, number of tiles run through the model)."""
    _, _, h, w = lr.shape
    lr = F.pad(lr, (0, max(0, TILE - w), 0, max(0, TILE - h)), mode="replicate")   # images smaller than a tile
    H, W = lr.shape[-2:]
    coords = [(y, x) for y in tile_starts(H) for x in tile_starts(W)]
    out = torch.zeros(1, 1, H * SCALE, W * SCALE, device=lr.device)
    hits = torch.zeros_like(out)
    t = TILE * SCALE
    for i in range(0, len(coords), BATCH_TILES):
        chunk = coords[i:i + BATCH_TILES]
        sr = model(torch.cat([lr[..., y:y + TILE, x:x + TILE] for y, x in chunk]))
        for (y, x), tile in zip(chunk, sr):
            out[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += tile
            hits[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += 1
    return (out / hits)[..., :h * SCALE, :w * SCALE], len(coords)


def shave(img):
    return img[..., SHAVE:-SHAVE, SHAVE:-SHAVE] if SHAVE else img


if SAVE_SR_IMAGES:
    from PIL import Image
    (OUTPUT_DIR / "sr").mkdir(parents=True, exist_ok=True)

rows = []
for image_id in tqdm(test_ids):
    hr = to_tensor(normalize_image(load_image(EMPS_DIR / "images" / f"{image_id}.png")))   # [1, H, W]
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]            # so the SR size matches the HR size
    lr = degrade(hr, filename=image_id, scale=SCALE)      # [1, H/4, W/4]

    sr, n_tiles = super_resolve(lr.unsqueeze(0).to(device))
    sr = sr.clamp(0, 1).cpu()
    bicubic = F.interpolate(lr.unsqueeze(0), scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)
    hr = hr.unsqueeze(0)

    rows.append({
        "image": image_id,
        "hr_size": f"{hr.shape[-1]}x{hr.shape[-2]}",
        "psnr": psnr(shave(sr), shave(hr)),
        "bicubic_psnr": psnr(shave(bicubic), shave(hr)),
        "ssim": 1.0 - ssim_loss(shave(sr), shave(hr)).item(),
        "bicubic_ssim": 1.0 - ssim_loss(shave(bicubic), shave(hr)).item(),
        "tiles": n_tiles,
        "gflops": n_tiles * FLOPS_PER_TILE / 1e9,
    })
    if SAVE_SR_IMAGES:
        arr = np.rint(sr[0, 0].numpy() * 65535).astype(np.uint16)
        Image.fromarray(arr).save(OUTPUT_DIR / "sr" / f"{image_id}_sr.tif", format="TIFF")

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "test_results.csv", index=False)
results.head()


## Results

In [ ]:
summary = {
    "test images": len(results),
    "average PSNR (dB)": round(results["psnr"].mean(), 3),
    "average bicubic PSNR (dB)": round(results["bicubic_psnr"].mean(), 3),
    "average SSIM": round(results["ssim"].mean(), 4),
    "average bicubic SSIM": round(results["bicubic_ssim"].mean(), 4),
    f"GFLOPs per {TILE}x{TILE} LR input": round(FLOPS_PER_TILE / 1e9, 3),
    "average GFLOPs per test image": round(results["gflops"].mean(), 3),
    "parameters": n_params,
}
for name, value in summary.items():
    print(f"{name:>32}: {value}")
pd.Series(summary).to_frame("value").to_csv(OUTPUT_DIR / "test_summary.csv")


## LR / SR examples
A few test images side by side: the degraded LR input (upscaled with nearest-neighbour so it is shown at the
same size) and the model's SR output, with the image's PSNR / SSIM.

In [ ]:
import matplotlib.pyplot as plt

N_EXAMPLES = 4   # how many test images to show

example_ids = test_ids[:N_EXAMPLES]
fig, axes = plt.subplots(len(example_ids), 2, figsize=(10, 5 * len(example_ids)), squeeze=False)
for row, image_id in zip(axes, example_ids):
    hr = to_tensor(normalize_image(load_image(EMPS_DIR / "images" / f"{image_id}.png")))
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]
    lr = degrade(hr, filename=image_id, scale=SCALE)
    sr, _ = super_resolve(lr.unsqueeze(0).to(device))
    sr = sr.clamp(0, 1).cpu()

    stats = results.set_index("image").loc[image_id]
    row[0].imshow(lr[0].numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    row[0].set_title(f"{image_id} — LR {lr.shape[-1]}x{lr.shape[-2]}")
    row[1].imshow(sr[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    row[1].set_title(f"SR {sr.shape[-1]}x{sr.shape[-2]} — "
                     f"PSNR {stats['psnr']:.2f} dB, SSIM {stats['ssim']:.4f}")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lr_sr_examples.png", dpi=150)
plt.show()


In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / "test_summary.csv"))
files.download(str(OUTPUT_DIR / "test_results.csv"))   # per-image PSNR / SSIM / FLOPs
